In [1]:
import os, gc, re, glob, time, difflib, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

warnings.filterwarnings('ignore')

# -------------------------------------------------------------------------
# 1. Configuration & Data Discovery
# -------------------------------------------------------------------------
cands = [
    os.environ.get('DATA_DIR', '.'),
    '.',
    '/kaggle/input/nexus-optiml-2026-operation-insight',
] + glob.glob('/kaggle/input/*/') + glob.glob('/kaggle/input/*/*/')

DATA_DIR = next((d for d in cands if os.path.exists(os.path.join(d, 'test.csv')) or os.path.exists(os.path.join(d, 'train.csv'))), '.')
SEED = 42
print(f"Data directory: {DATA_DIR} | XGBoost: {HAS_XGB} | CatBoost: {HAS_CB}")


# -------------------------------------------------------------------------
# 2. String Normalization & Vectorized Lookups
# -------------------------------------------------------------------------
def canon_map(values, canon, cutoff=0.75):
    key = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
    ck = {key(c): c for c in canon}
    out = {}
    for v in pd.unique(values):
        if pd.isna(v):
            out[v] = 'unknown'
            continue
        k = key(v)
        if k in ck:
            out[v] = ck[k]
        else:
            r = difflib.get_close_matches(k, list(ck.keys()), n=1, cutoff=cutoff)
            out[v] = ck[r[0]] if r else str(v).strip().lower()
    return out


def fast_mode_map(df, group_col, target_col):
    """Vectorized categorical mode lookup (replaces slow groupby.apply)."""
    valid = df[[group_col, target_col]].dropna()
    counts = valid.value_counts().reset_index(name='n')
    top = counts.sort_values('n', ascending=False).drop_duplicates(group_col)
    return top.set_index(group_col)[target_col]


# -------------------------------------------------------------------------
# 3. Clean Supplementary Reference Tables
# -------------------------------------------------------------------------
def clean_tables(pr, st, cu, cal):
    # Products: canonical categories + physical bounds
    vc = pr['category'].value_counts()
    top_cats = [c for c, n in vc.items() if n >= 300 and c == c.lower().strip() and ' ' not in c]
    pr['category'] = pr['category'].map(canon_map(pr['category'].unique(), top_cats))

    for d_col in ['length_cm', 'width_cm', 'height_cm', 'list_price']:
        pr[d_col] = pr[d_col].clip(lower=0.1, upper=pr[d_col].quantile(0.999))

    pr['vol'] = pr['length_cm'] * pr['width_cm'] * pr['height_cm']
    pr['weight_missing'] = (pr['weight_kg'].isnull() | (pr['weight_kg'] <= 0)).astype('int8')
    pr.loc[pr['weight_kg'] <= 0, 'weight_kg'] = np.nan
    pr['weight_kg'] = pr['weight_kg'].fillna(
        pr.groupby(['category', 'packaging_type'])['weight_kg'].transform('median')
    ).fillna(pr['weight_kg'].median()).clip(upper=pr['weight_kg'].quantile(0.999))

    pr['density'] = (pr['weight_kg'] / (pr['vol'] + 1.0)).clip(upper=0.05)
    pr['price_to_cat_med'] = (pr['list_price'] / pr.groupby('category')['list_price'].transform('median')).clip(0.05, 20.0)
    pr['log_list_price'] = np.log1p(pr['list_price'])
    pr['log_reviews'] = np.log1p(pr['review_count'].clip(lower=0))
    pr['launch_date'] = pd.to_datetime(pr['launch_date'])

    # Stores: format medians for physical sizes
    clean_regions = [r for r, n in st['region'].value_counts().items() if n >= 5]
    st['region'] = st['region'].map(canon_map(st['region'].unique(), clean_regions))
    st['size_missing'] = st['size_sqm'].isnull().astype('int8')
    st['size_sqm'] = st['size_sqm'].fillna(st.groupby('format')['size_sqm'].transform('median')).fillna(0)
    st['opened_date'] = pd.to_datetime(st['opened_date'])
    st['is_physical'] = st['channel'].isin(['store', 'catalog']).astype('int8')

    # Customers: deduplication + tier progression
    cu['region'] = cu['region'].map(canon_map(cu['region'].unique(), clean_regions))
    cu['record_updated_at'] = pd.to_datetime(cu['record_updated_at'])
    cu = cu.sort_values('record_updated_at')

    n_rec = cu.groupby('customer_id').size().rename('n_records').astype('int8')
    first_tier = cu.drop_duplicates('customer_id', keep='first').set_index('customer_id')['loyalty_tier']
    cu['age_band'] = cu.groupby('customer_id')['age_band'].ffill()
    cu = cu.drop_duplicates('customer_id', keep='last').copy()

    cu['n_records'] = cu['customer_id'].map(n_rec)
    cu['tier_first'] = cu['customer_id'].map(first_tier)
    cu['tier_changed'] = (cu['tier_first'] != cu['loyalty_tier']).astype('int8')
    cu['age_missing'] = cu['age_band'].isnull().astype('int8')

    age_modes = cu.dropna(subset=['age_band']).groupby(['customer_type', 'loyalty_tier'])['age_band'].agg(lambda s: s.value_counts().index[0])
    mode_series = pd.Series(cu.set_index(['customer_type', 'loyalty_tier']).index.map(age_modes), index=cu.index)
    cu['age_band'] = cu['age_band'].fillna(mode_series).fillna('25-34')
    cu['signup_date'] = pd.to_datetime(cu['signup_date'])

    # Calendar: stationary diffs
    cal['week_start'] = pd.to_datetime(cal['week_start'])
    cal = cal.sort_values('week_start')
    for c in ['cpi', 'fuel_price', 'unemployment']:
        cal[c + '_chg'] = cal[c].diff().fillna(0)
    cal['temp_dev'] = cal['temperature_f'] - cal['temperature_f'].rolling(52, min_periods=1, center=True).mean()

    return pr, st, cu, cal


# -------------------------------------------------------------------------
# 4. Feature Pipeline with Smart Outlier Handling
# -------------------------------------------------------------------------
def build_dataset(tr, te, pr, st, cu, cal):
    t0 = time.time()
    pr, st, cu, cal = clean_tables(pr, st, cu, cal)

    tr['is_train'] = 1
    te['is_train'] = 0
    te['returned'] = np.nan
    df = pd.concat([tr, te], ignore_index=True)
    del tr, te; gc.collect()

    df['ts'] = pd.to_datetime(df['order_ts'])
    df['week_start'] = df['ts'].dt.normalize() - pd.to_timedelta(df['ts'].dt.dayofweek, unit='D')

    df = df.merge(cu.drop(columns=['record_updated_at']).rename(columns={'region': 'c_region', 'customer_type': 'c_type'}), on='customer_id', how='left')
    df = df.merge(pr.rename(columns={'category': 'cat'}), on='product_id', how='left')
    df = df.merge(st.rename(columns={'channel': 's_channel', 'region': 's_region', 'format': 's_format'}), on='store_id', how='left')
    df = df.merge(cal[['week_start', 'holiday_flag', 'temperature_f', 'fuel_surcharge', 'cpi_chg', 'fuel_price_chg', 'unemployment_chg', 'temp_dev']], on='week_start', how='left')

    # Outlier Step 1: Repair 100x cents-vs-dollars unit_price error (~2.0% of data)
    raw_lv = df['unit_price'] * df['quantity']
    raw_ov_ratio = df['order_value'] / (raw_lv + 1e-5)
    raw_up_lp = df['unit_price'] / (df['list_price'] + 1e-5)

    is_up_100x = ((raw_ov_ratio > 0.004) & (raw_ov_ratio < 0.025)) | (raw_up_lp > 10.0)
    df['up_100x_flag'] = is_up_100x.astype('int8')
    df.loc[is_up_100x, 'unit_price'] /= 100.0

    # Outlier Step 2: Repair 10x quantity multiplier error (~0.1% of data)
    raw_lv = df['unit_price'] * df['quantity']
    raw_ov_ratio = df['order_value'] / (raw_lv + 1e-5)
    raw_up_lp = df['unit_price'] / (df['list_price'] + 1e-5)

    is_qty_10x = (raw_ov_ratio > 0.05) & (raw_ov_ratio < 0.20) & (raw_up_lp < 2.0) & (df['quantity'] % 10 == 0)
    df['qty_10x_flag'] = is_qty_10x.astype('int8')
    df.loc[is_qty_10x, 'quantity'] = (df.loc[is_qty_10x, 'quantity'] // 10).clip(lower=1)

    # Outlier Step 3: Impute missing discounts algebraically
    df['discount_missing'] = df['discount'].isnull().astype('int8')
    implied_disc = (1.0 - (df['unit_price'] / df['list_price'])).clip(0.0, 0.85)
    df['discount'] = df['discount'].fillna(implied_disc).clip(0.0, 0.85)
    df['disc_gap'] = (df['discount'] - implied_disc).clip(-0.5, 0.5)

    # Missing flags & fast modal categorical imputation
    df['freight_missing'] = df['freight_cost'].isnull().astype('int8')
    df['clicks_missing'] = df['session_clicks'].isnull().astype('int8')

    df['ship_mode'] = df['ship_mode'].fillna(df['channel'].map({'store': 'in_store'})).fillna('standard')
    cust_pm_map = fast_mode_map(df, 'customer_id', 'payment_method')
    ch_pm_map = fast_mode_map(df, 'channel', 'payment_method')
    df['payment_method'] = df['payment_method'].fillna(df['customer_id'].map(cust_pm_map)).fillna(df['channel'].map(ch_pm_map)).fillna('card')
    df['tz_hint'] = df['tz_hint'].fillna('unknown')

    nosess = df['device_type'].isin(['phone', 'pos'])
    df.loc[nosess, 'session_clicks'] = df.loc[nosess, 'session_clicks'].fillna(0)
    df['session_clicks'] = df['session_clicks'].fillna(df.groupby(['device_type', 'channel'])['session_clicks'].transform('median')).clip(0, 30)

    df['freight_cost'] = df['freight_cost'].fillna(df.groupby(['channel', 'ship_mode'])['freight_cost'].transform('median')).fillna(df['freight_cost'].median())
    df['freight_cost'] = df['freight_cost'].clip(0.0, df['freight_cost'].quantile(0.999))

    # Cleaned value & ratio interactions
    df['lv'] = df['unit_price'] * df['quantity']
    df['lq'] = np.log1p(df['quantity'])
    df['log_lv'] = np.log1p(df['lv'])
    df['log_price'] = np.log1p(df['unit_price'])
    df['log_ov'] = np.log1p(df['order_value'].clip(lower=0))
    df['log_freight'] = np.log1p(df['freight_cost'])

    df['price_ratio'] = (df['unit_price'] / df['list_price']).clip(0.05, 2.5)
    df['log_price_ratio'] = np.log(df['price_ratio'])
    df['ov_ratio'] = (df['order_value'] / (df['lv'] + 1e-4)).clip(0.5, 1.5)
    df['ov_diff'] = (df['order_value'] - df['lv']).clip(-10.0, 10.0)
    df['undiscounted_lv'] = df['list_price'] * df['quantity']
    df['saved_amount'] = np.log1p((df['undiscounted_lv'] - df['lv']).clip(lower=0))
    df['freight_per_unit'] = (df['freight_cost'] / df['quantity']).clip(0, 100)
    df['freight_share'] = (df['freight_cost'] / (df['lv'] + 1.0)).clip(0, 5.0)
    df['freight_per_kg'] = (df['freight_cost'] / (df['weight_kg'] * df['quantity'] + 0.1)).clip(0, 100)
    df['total_weight_kg'] = np.log1p(df['weight_kg'] * df['quantity'])

    # Temporal & session patterns
    df['hour'] = df['ts'].dt.hour
    df['dow'] = df['ts'].dt.dayofweek
    df['month'] = df['ts'].dt.month
    df['day_of_year'] = df['ts'].dt.dayofyear
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['date_only'] = ((df['ts'].dt.hour == 0) & (df['ts'].dt.minute == 1)).astype('int8')

    df['window_x_price'] = df['return_window_days'] * df['log_price']
    df['has_coupon'] = df['coupon_code'].notnull().astype('int8')
    df['clicks_per_item'] = (df['session_clicks'] / df['items_in_order'].clip(lower=1)).clip(0, 30)
    df['line_pos'] = (df['line_no'] / df['items_in_order'].clip(lower=1)).clip(0, 1.5)

    # Geographic alignments
    df['cross_region'] = (df['c_region'] != df['s_region']).astype('int8')
    df['cust_country'] = df['c_region'].str.split('-').str[0]
    df['store_country'] = df['s_region'].str.split('-').str[0]
    df['cross_country'] = (df['cust_country'] != df['store_country']).astype('int8')
    df['days_since_signup'] = (df['ts'] - df['signup_date']).dt.days.clip(-30, 5000)
    df['prod_age'] = (df['ts'] - df['launch_date']).dt.days.clip(-30, 5000)
    df['store_age'] = (df['ts'] - df['opened_date']).dt.days.clip(-30, 10000)

    # Order context aggregations
    g = df.groupby('order_id')
    df['o_lines'] = g['order_line_id'].transform('size')
    df['o_qty'] = g['quantity'].transform('sum')
    df['o_nprod'] = g['product_id'].transform('nunique')
    df['o_ncat'] = g['cat'].transform('nunique')
    df['o_lv'] = g['lv'].transform('sum')
    df['log_o_lv'] = np.log1p(df['o_lv'])
    df['o_cover'] = (df['o_lines'] / df['items_in_order'].clip(lower=1)).clip(0, 2.0)
    df['qty_share'] = df['quantity'] / df['o_qty'].clip(lower=1)
    df['val_share'] = df['lv'] / (df['o_lv'] + 1e-4)
    df['price_rank'] = g['unit_price'].rank(pct=True)
    df['o_maxprice'] = np.log1p(g['unit_price'].transform('max'))
    df['o_minprice'] = np.log1p(g['unit_price'].transform('min'))
    df['o_mean_disc'] = g['discount'].transform('mean')

    df = df.copy()

    # Entity statistics
    for c in ['customer_id', 'product_id', 'store_id', 'supplier_code']:
        df[c + '_n'] = np.log1p(df.groupby(c)['order_line_id'].transform('size'))

    df['is_mkt'] = (df['s_channel'] == 'marketplace').astype('int8')

    gc_ = df.groupby('customer_id')
    df['cust_orders'] = np.log1p(gc_['order_id'].transform('nunique'))
    df['cust_mean_lv'] = gc_['log_lv'].transform('mean')
    df['cust_std_lv'] = gc_['log_lv'].transform('std').fillna(0)
    df['cust_lv_diff'] = df['log_lv'] - df['cust_mean_lv']
    df['cust_mean_disc'] = gc_['discount'].transform('mean')
    df['cust_n_cat'] = gc_['cat'].transform('nunique')
    df['cust_mkt_share'] = gc_['is_mkt'].transform('mean')
    df['cust_mean_ratio'] = gc_['log_price_ratio'].transform('mean')
    df['cust_mean_qty'] = gc_['lq'].transform('mean')
    df['cust_gift_rate'] = gc_['gift_wrap'].transform('mean')

    gp = df.groupby('product_id')
    df['prod_mean_ratio'] = gp['log_price_ratio'].transform('mean')
    df['prod_mean_qty'] = gp['lq'].transform('mean')
    df['prod_n_cust'] = np.log1p(gp['customer_id'].transform('nunique'))
    df['prod_mean_disc'] = gp['discount'].transform('mean')
    df['prod_price_vs_mean'] = df['log_price_ratio'] - df['prod_mean_ratio']

    gs = df.groupby('store_id')
    df['store_mean_lv'] = gs['log_lv'].transform('mean')
    df['store_mean_disc'] = gs['discount'].transform('mean')
    df['store_n_prod'] = np.log1p(gs['product_id'].transform('nunique'))

    df['cust_x_store_n'] = df.groupby(['customer_id', 'store_id'])['order_line_id'].transform('size')
    df['cust_x_cat_n'] = df.groupby(['customer_id', 'cat'])['order_line_id'].transform('size')

    # Inter-order time intervals WITHOUT sorting df in-place (maintains original row order)
    order_ts_first = df.drop_duplicates('order_id')[['customer_id', 'order_id', 'ts']].sort_values(['customer_id', 'ts'])
    order_ts_first['prev_order_gap'] = order_ts_first.groupby('customer_id')['ts'].diff().dt.total_seconds() / 86400.0
    df = df.merge(order_ts_first[['order_id', 'prev_order_gap']], on='order_id', how='left')
    df['is_first_order'] = df['prev_order_gap'].isnull().astype('int8')
    df['prev_order_gap'] = df['prev_order_gap'].fillna(-1.0).clip(-1.0, 730.0)

    CATS = [
        'channel', 'payment_method', 'ship_mode', 'device_type', 'warehouse_zone',
        'picker_shift', 'tz_hint', 'cat', 'c_type', 'age_band', 'loyalty_tier',
        'tier_first', 's_format', 's_channel', 's_region', 'c_region',
        'packaging_type', 'color', 'preferred_contact', 'cust_country', 'store_country'
    ]
    for c in CATS:
        df[c] = df[c].astype(str).astype('category').cat.codes.astype('int16')

    print(f"Dataset compiled in {time.time() - t0:.1f}s | Shape: {df.shape}")
    return df, CATS


# -------------------------------------------------------------------------
# 5. Vectorized Out-of-Fold Target Encoding
# -------------------------------------------------------------------------
TE_SPECS = [
    ('customer_id', 30.0),
    ('product_id', 25.0),
    ('store_id', 50.0),
    ('supplier_code', 30.0),
    ('banner_id', 50.0),
]

def fast_target_encode(tr_df, y_tr, val_df, n_splits=5):
    prior = float(np.mean(y_tr))
    oof_df = pd.DataFrame(index=np.arange(len(tr_df)))
    val_te_df = pd.DataFrame(index=np.arange(len(val_df)))

    for col, k in TE_SPECS:
        oof_df[f'{col}_te'] = np.full(len(tr_df), prior, dtype='float32')

    gkf = GroupKFold(n_splits=n_splits)
    temp = tr_df[[c for c, _ in TE_SPECS]].copy()
    temp['_y'] = y_tr

    full_stats = {col: temp.groupby(col)['_y'].agg(['sum', 'count']) for col, _ in TE_SPECS}

    for _, val_idx in gkf.split(tr_df, y_tr, groups=tr_df['order_id'].values):
        fold_part = temp.iloc[val_idx]
        for col, k in TE_SPECS:
            fg = fold_part.groupby(col)['_y'].agg(['sum', 'count'])
            tot = full_stats[col].reindex(fg.index)
            rem_sum = tot['sum'] - fg['sum']
            rem_cnt = tot['count'] - fg['count']
            enc_map = (rem_sum + k * prior) / (rem_cnt + k)
            oof_df.loc[val_idx, f'{col}_te'] = fold_part[col].map(enc_map).fillna(prior).values.astype('float32')

    for col, k in TE_SPECS:
        g = full_stats[col]
        enc_map = (g['sum'] + k * prior) / (g['count'] + k)
        val_te_df[f'{col}_te'] = val_df[col].map(enc_map).fillna(prior).values.astype('float32')

    return oof_df, val_te_df


# -------------------------------------------------------------------------
# 6. Load Data & Prepare Modeling Matrices
# -------------------------------------------------------------------------
tr = pd.read_csv(f'{DATA_DIR}/train.csv')
te = pd.read_csv(f'{DATA_DIR}/test.csv')
pr = pd.read_csv(f'{DATA_DIR}/products.csv')
st = pd.read_csv(f'{DATA_DIR}/stores.csv')
cu = pd.read_csv(f'{DATA_DIR}/customers.csv')
cal = pd.read_csv(f'{DATA_DIR}/calendar.csv')

df, CATS = build_dataset(tr, te, pr, st, cu, cal)
del pr, st, cu, cal; gc.collect()

DROP_COLS = [
    'order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id',
    'order_ts', 'returned', 'is_train', 'ts', 'week_start', 'signup_date',
    'launch_date', 'opened_date', 'coupon_code', 'supplier_code', 'promo_channel_code'
]
FEATS = [c for c in df.columns if c not in DROP_COLS]
for c in FEATS:
    if df[c].dtype == 'float64':
        df[c] = df[c].astype('float32')

T = df[df['is_train'] == 1].reset_index(drop=True)
TEST = df[df['is_train'] == 0].reset_index(drop=True)
y_all = T['returned'].values.astype('int32')
del df; gc.collect()

IN_CATS = [c for c in FEATS if c in CATS]
print(f"Train lines: {len(T):,} | Test lines: {len(TEST):,} | Total features: {len(FEATS)}")


def get_time_weights(ts_series):
    days = (ts_series - ts_series.min()).dt.total_seconds() / 86400.0
    rk = days / (days.max() + 1e-5)
    return (0.5 + 0.5 * rk).values.astype('float32')


LGB_PARAMS = dict(
    objective='binary', metric='auc', learning_rate=0.045,
    num_leaves=63, max_depth=8, min_child_samples=250,
    feature_fraction=0.75, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=2.0, lambda_l2=15.0, max_bin=255,
    verbose=-1, n_jobs=-1, random_state=SEED
)

XGB_PARAMS = dict(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    learning_rate=0.045, max_depth=7, min_child_weight=200,
    subsample=0.8, colsample_bytree=0.75, reg_lambda=15.0, reg_alpha=2.0,
    nthread=-1, seed=SEED
)

CB_PARAMS = dict(
    loss_function='Logloss', eval_metric='AUC', learning_rate=0.065,
    depth=7, l2_leaf_reg=12.0, random_seed=SEED, thread_count=-1, verbose=False
)

# -------------------------------------------------------------------------
# 7. Time-Split Cross-Validation
# -------------------------------------------------------------------------
VAL_START = '2024-11-01'
tr_mask = (T['ts'] < VAL_START).values
va_mask = (T['ts'] >= VAL_START).values

A, B = T[tr_mask].reset_index(drop=True), T[va_mask].reset_index(drop=True)
yA, yB = y_all[tr_mask], y_all[va_mask]
wA = get_time_weights(A['ts'])

teA, teB = fast_target_encode(A, yA, B)
XA = pd.concat([A[FEATS], teA], axis=1)
XB = pd.concat([B[FEATS], teB], axis=1)
del A, B, teA, teB; gc.collect()

val_preds, best_iters = {}, {}

# LightGBM validation
dtrain_lgb = lgb.Dataset(XA, label=yA, weight=wA, categorical_feature=IN_CATS, free_raw_data=False)
dval_lgb = lgb.Dataset(XB, label=yB, reference=dtrain_lgb, categorical_feature=IN_CATS, free_raw_data=False)
m_lgb = lgb.train(
    LGB_PARAMS, dtrain_lgb, num_boost_round=3500,
    valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(150, verbose=False)]
)
val_preds['lgb'] = m_lgb.predict(XB, num_iteration=m_lgb.best_iteration)
best_iters['lgb'] = m_lgb.best_iteration
print(f"Validation LightGBM AUC: {roc_auc_score(yB, val_preds['lgb']):.5f} (rounds={best_iters['lgb']})")

# XGBoost validation
if HAS_XGB:
    dtrain_xgb = xgb.DMatrix(XA, label=yA, weight=wA)
    dval_xgb = xgb.DMatrix(XB, label=yB)
    m_xgb = xgb.train(
        XGB_PARAMS, dtrain_xgb, num_boost_round=3000,
        evals=[(dval_xgb, 'val')], early_stopping_rounds=150, verbose_eval=False
    )
    val_preds['xgb'] = m_xgb.predict(dval_xgb, iteration_range=(0, m_xgb.best_iteration + 1))
    best_iters['xgb'] = m_xgb.best_iteration + 1
    print(f"Validation XGBoost  AUC: {roc_auc_score(yB, val_preds['xgb']):.5f} (rounds={best_iters['xgb']})")
    del dtrain_xgb, dval_xgb; gc.collect()

# CatBoost validation
if HAS_CB:
    m_cb = CatBoostClassifier(iterations=2000, early_stopping_rounds=120, **CB_PARAMS)
    m_cb.fit(XA, yA, sample_weight=wA, eval_set=(XB, yB), use_best_model=True)
    val_preds['cb'] = m_cb.predict_proba(XB)[:, 1]
    best_iters['cb'] = m_cb.get_best_iteration() + 1
    print(f"Validation CatBoost AUC: {roc_auc_score(yB, val_preds['cb']):.5f} (rounds={best_iters['cb']})")

del XA, XB, dtrain_lgb, dval_lgb; gc.collect()

# Optimize rank-averaging weights
rk = lambda x: rankdata(x) / len(x)
names = list(val_preds.keys())
best_w, best_auc = None, 0.0
import itertools
for w in itertools.product(np.linspace(0, 1, 21), repeat=len(names)):
    if abs(sum(w) - 1.0) > 1e-5:
        continue
    blend_val = sum(wi * rk(val_preds[n]) for wi, n in zip(w, names))
    score = roc_auc_score(yB, blend_val)
    if score > best_auc:
        best_auc, best_w = score, w

W = dict(zip(names, best_w))
print(f"Optimal Model Weights: {W} | Time-Holdout Blend AUC: {best_auc:.5f}")


# -------------------------------------------------------------------------
# 8. Full Retraining & Test Prediction
# -------------------------------------------------------------------------
w_full = get_time_weights(T['ts'])
teF, teT = fast_target_encode(T, y_all, TEST)
XF = pd.concat([T[FEATS], teF], axis=1)
XT = pd.concat([TEST[FEATS], teT], axis=1)
del T, teF, teT; gc.collect()

test_preds = {}

# 3-Seed LightGBM
lgb_rounds = int(best_iters['lgb'] * 1.12)
d_full_lgb = lgb.Dataset(XF, label=y_all, weight=w_full, categorical_feature=IN_CATS, free_raw_data=False)
lgb_bag = []
for s in range(3):
    p = dict(LGB_PARAMS, random_state=SEED + s * 11)
    m = lgb.train(p, d_full_lgb, num_boost_round=lgb_rounds)
    lgb_bag.append(rk(m.predict(XT)))
test_preds['lgb'] = np.mean(lgb_bag, axis=0)
del d_full_lgb; gc.collect()

# XGBoost Full Fit
if HAS_XGB and W.get('xgb', 0) > 0:
    xgb_rounds = int(best_iters['xgb'] * 1.12)
    d_full_xgb = xgb.DMatrix(XF, label=y_all, weight=w_full)
    m = xgb.train(XGB_PARAMS, d_full_xgb, num_boost_round=xgb_rounds)
    test_preds['xgb'] = rk(m.predict(xgb.DMatrix(XT)))
    del d_full_xgb; gc.collect()

# CatBoost Full Fit
if HAS_CB and W.get('cb', 0) > 0:
    cb_rounds = int(best_iters['cb'] * 1.12)
    m = CatBoostClassifier(iterations=cb_rounds, **CB_PARAMS)
    m.fit(XF, y_all, sample_weight=w_full)
    test_preds['cb'] = rk(m.predict_proba(XT)[:, 1])

final_model_pred = sum(W[n] * test_preds[n] for n in test_preds)
sub_opt = pd.DataFrame({
    'order_line_id': TEST['order_line_id'].values,
    'returned': final_model_pred
})
sub_opt.to_csv('submission_optimized.csv', index=False)
print("Saved standalone model predictions to submission_optimized.csv")


# -------------------------------------------------------------------------
# 9. Automatic Key-Based Ensembling with Previous Best Submission
# -------------------------------------------------------------------------
y1_candidates = ['submission_y1.csv', f'{DATA_DIR}/submission_y1.csv', '/kaggle/working/submission_y1.csv']
y1_path = next((p for p in y1_candidates if os.path.exists(p)), None)

if y1_path is not None:
    print(f"Found previous best submission at {y1_path}! Aligning on order_line_id...")
    sub_y1 = pd.read_csv(y1_path)
    
    # Keyed mapping guarantees exact row alignment regardless of order
    sub_y1_map = sub_y1.set_index('order_line_id')['returned']
    sub_y1_aligned = sub_opt['order_line_id'].map(sub_y1_map).values
    
    rk_y1 = rankdata(sub_y1_aligned) / len(sub_y1_aligned)
    rk_opt = rankdata(sub_opt['returned']) / len(sub_opt)
    
    # 65% clean pipeline + 35% baseline
    final_blend = 0.65 * rk_opt + 0.35 * rk_y1
    
    final_sub = pd.DataFrame({
        'order_line_id': sub_opt['order_line_id'].values,
        'returned': final_blend
    })
    final_sub.to_csv('submission5.csv', index=False)
    print("Successfully generated final blended submission5.csv!")
else:
    sub_opt.to_csv('submission5.csv', index=False)
    print("Saved submission5.csv directly from the optimized model ensemble.")

Data directory: . | XGBoost: True | CatBoost: True
Dataset compiled in 31.3s | Shape: (1880000, 154)
Train lines: 1,400,005 | Test lines: 479,995 | Total features: 138
Validation LightGBM AUC: 0.71329 (rounds=601)
Validation XGBoost  AUC: 0.71161 (rounds=879)
Validation CatBoost AUC: 0.72329 (rounds=947)
Optimal Model Weights: {'lgb': np.float64(0.0), 'xgb': np.float64(0.0), 'cb': np.float64(1.0)} | Time-Holdout Blend AUC: 0.72329
Saved standalone model predictions to submission_optimized.csv
Found previous best submission at submission_y1.csv! Aligning on order_line_id...
Successfully generated final blended submission5.csv!
